Install Required Libraries

In [2]:
# Cell 1: Install Required Libraries
!pip install -q --upgrade langchain langchain-community langchain-huggingface faiss-cpu pypdf reportlab sentence-transformers transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests

Generate Sample PDF & Load Knowledge Base

In [3]:
# Cell 2: Generate Sample PDF & Load Knowledge Base
import os
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from langchain_community.document_loaders import PyPDFLoader

# 1. Helper function to generate a sample medical/technical PDF document
def create_sample_pdf(filename="sample_knowledge_base.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    c.drawString(100, 750, "Medical First Aid Knowledge Base - Task 5")
    c.drawString(100, 720, "1. Minor Burns: Cool the burn under cool running water for 10-15 minutes.")
    c.drawString(100, 700, "   Do not apply ice directly. Cover with a clean, non-stick bandage.")
    c.drawString(100, 670, "2. Cuts and Scrapes: Wash thoroughly with water and mild soap.")
    c.drawString(100, 650, "   Apply an antiseptic cream and cover with a sterile bandage.")
    c.drawString(100, 620, "3. Seasonal Allergies: Common symptoms include sneezing and itchy eyes.")
    c.drawString(100, 600, "   Stay hydrated and avoid known environmental allergens.")
    c.drawString(100, 570, "4. Daily Hydration: Adults should drink 2 to 3 liters of water daily.")
    c.save()

# 2. Automatically create the knowledge base PDF file
pdf_file_path = "sample_knowledge_base.pdf"
create_sample_pdf(pdf_file_path)

# 3. Load the PDF document using LangChain PyPDFLoader
loader = PyPDFLoader(pdf_file_path)
documents = loader.load()

print(f"✅ Created and loaded '{pdf_file_path}' successfully ({len(documents)} page(s)).")

/tmp/ipykernel_1367/1026573111.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ Created and loaded 'sample_knowledge_base.pdf' successfully (1 page(s)).


Text Chunking & Vector Database Setup (FAISS)

In [4]:
# Cell 3: Text Chunking & Vector Database Setup (FAISS)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Split text into manageable chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
docs = text_splitter.split_documents(documents)

# 2. Initialize HuggingFace embeddings model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 3. Create FAISS Vector Database index
vectorstore = FAISS.from_documents(docs, embeddings)

# 4. Define retriever object
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"✅ Vector store index created successfully with {len(docs)} text chunk(s)!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector store index created successfully with 2 text chunk(s)!


Load LLM Generator Model (TinyLlama)

In [5]:
# Cell 4: Load LLM Generator Model (TinyLlama)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

# HuggingFace Model Identifier
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load Tokenizer & Model with Float16 precision
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Create Hugging Face text generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=120,
    temperature=0.1,
    repetition_penalty=1.1,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)

print("✅ LLM Generator model loaded successfully!")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ LLM Generator model loaded successfully!


Build & Execute LangChain RAG Chain (LCEL)

In [7]:
# Cell 5: Build & Execute LangChain RAG Chain (LCEL)
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Define prompt template using LangChain PromptTemplate
prompt_template = """<|system|>
You are a helpful support assistant. Use the following retrieved context to answer the user's question accurately and concisely.

Context:
{context}</s>
<|user|>
Question: {question}</s>
<|assistant|>
Answer:"""

prompt = PromptTemplate.from_template(prompt_template)

# Helper function to join retrieved page content
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 2. Construct Modern LangChain Expression Language (LCEL) RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 3. Test Question
query = "What should i do if i want to study AI Automation?"

print("❓ Query:", query)
print("\n🔍 Retrieving context and generating response via LangChain...\n")

response = rag_chain.invoke(query)

print("=========================================")
print("🤖 LANGCHAIN RAG RESPONSE:")
print("=========================================")
print(response.strip())
print("=========================================")

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❓ Query: What should i do if i want to study AI Automation?

🔍 Retrieving context and generating response via LangChain...

🤖 LANGCHAIN RAG RESPONSE:
If you want to study Artificial Intelligence (AI) Automation, here are some steps you can follow:

1. Research and learn about AI and its applications in various industries.
2. Choose a course or program that focuses on AI and automation.
3. Enroll in a reputable university or college that offers courses in AI and automation.
4. Attend workshops, seminars, and conferences related to AI and automation.
5. Network with professionals in the field and connect
